In [ ]:
%pip install -q qiskit qiskit-aer pylatexenc scikit-learn pandas matplotlib

In [ ]:
# ============================================================
# PERSON 2: QUANTUM CIRCUIT DESIGN
# Entangled vs Non-Entangled Circuits
# ============================================================

import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector

print("=" * 60)
print("PERSON 2 - QUANTUM CIRCUIT DESIGN")
print("=" * 60)


# 1. PREPARE IRIS DATA
iris = load_iris()
X = iris.data
y = iris.target

# Keep Setosa and Versicolor only
mask = (y == 0) | (y == 1)
X = X[mask][:, :2]
y = y[mask]

# 80:20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Scale using training data only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

angle_scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_quantum = np.clip(
    angle_scaler.fit_transform(X_train), 0, np.pi
)
X_test_quantum = np.clip(
    angle_scaler.transform(X_test), 0, np.pi
)

print("\nDATA VALIDATION")
print("Training shape:", X_train_quantum.shape)
print("Testing shape :", X_test_quantum.shape)
print("Training labels:", y_train.shape)
print("Testing labels :", y_test.shape)
print("Angle range:", X_train_quantum.min(), "to",
      X_train_quantum.max())


# 2. CREATE 2-QUBIT NON-ENTANGLED CIRCUIT
x = ParameterVector("x", 2)
theta = ParameterVector("theta", 2)

qc_2_non = QuantumCircuit(2)
qc_2_non.ry(x[0], 0)
qc_2_non.ry(x[1], 1)
qc_2_non.ry(theta[0], 0)
qc_2_non.ry(theta[1], 1)


# 3. CREATE 2-QUBIT ENTANGLED CIRCUIT
xe = ParameterVector("x", 2)
te = ParameterVector("theta", 2)

qc_2_ent = QuantumCircuit(2)
qc_2_ent.ry(xe[0], 0)
qc_2_ent.ry(xe[1], 1)
qc_2_ent.ry(te[0], 0)
qc_2_ent.ry(te[1], 1)
qc_2_ent.cx(0, 1)


# 4. DISPLAY CIRCUITS
print("\n2-QUBIT NON-ENTANGLED CIRCUIT")
print(qc_2_non.draw(output="text"))

print("\n2-QUBIT ENTANGLED CIRCUIT")
print(qc_2_ent.draw(output="text"))


# 5. OPTIONAL CIRCUIT IMAGES
# If the Matplotlib drawer fails, the program continues.
try:
    import matplotlib.pyplot as plt

    for circuit, filename, title in [
        (qc_2_non, "2qubit_non_entangled.png",
         "2-Qubit Non-Entangled Circuit"),
        (qc_2_ent, "2qubit_entangled.png",
         "2-Qubit Entangled Circuit")
    ]:
        fig = circuit.draw(output="mpl")
        fig.suptitle(title)
        fig.savefig(filename, dpi=200, bbox_inches="tight")
        plt.show()
        plt.close(fig)

    print("\n2-qubit circuit images saved.")

except Exception as error:
    print("\nGraphical circuit images unavailable:", error)
    print("Text circuit diagrams above are available.")


# 6. CIRCUIT STATISTICS
def get_statistics(circuit):
    counts = circuit.count_ops()
    return {
        "Qubits": circuit.num_qubits,
        "Total Gates": sum(counts.values()),
        "Single-Qubit Gates": sum(
            instruction.operation.num_qubits == 1
            for instruction in circuit.data
        ),
        "Two-Qubit Gates": sum(
            instruction.operation.num_qubits == 2
            for instruction in circuit.data
        ),
        "CNOT Gates": counts.get("cx", 0),
        "Circuit Depth": circuit.depth()
    }


# 7. CREATE 4-QUBIT DEMONSTRATION CIRCUITS
x4 = ParameterVector("x4", 4)
t4 = ParameterVector("theta4", 4)

qc_4_non = QuantumCircuit(4)

for i in range(4):
    qc_4_non.ry(x4[i], i)
    qc_4_non.ry(t4[i], i)

x4e = ParameterVector("x4", 4)
t4e = ParameterVector("theta4", 4)

qc_4_ent = QuantumCircuit(4)

for i in range(4):
    qc_4_ent.ry(x4e[i], i)
    qc_4_ent.ry(t4e[i], i)

qc_4_ent.cx(0, 1)
qc_4_ent.cx(1, 2)
qc_4_ent.cx(2, 3)

print("\n4-QUBIT NON-ENTANGLED CIRCUIT")
print(qc_4_non.draw(output="text"))

print("\n4-QUBIT ENTANGLED CIRCUIT")
print(qc_4_ent.draw(output="text"))


# 8. COMPARISON TABLE
circuits = {
    "2-Qubit Non-Entangled": qc_2_non,
    "2-Qubit Entangled": qc_2_ent,
    "4-Qubit Non-Entangled": qc_4_non,
    "4-Qubit Entangled": qc_4_ent
}

rows = []

for name, circuit in circuits.items():
    row = {"Circuit": name}
    row.update(get_statistics(circuit))
    rows.append(row)

comparison = pd.DataFrame(rows)

print("\nCIRCUIT COMPARISON")
print(comparison.to_string(index=False))

comparison.to_csv(
    "person2_circuit_comparison.csv",
    index=False
)

print("\nSaved: person2_circuit_comparison.csv")


# 9. FINISH
print("\n" + "=" * 60)
print("PERSON 2 CIRCUIT DESIGN COMPLETED")
print("=" * 60)
print("Completed:")
print("1. Two-qubit non-entangled circuit")
print("2. Two-qubit circuit with CNOT")
print("3. Four-qubit demonstration circuits")
print("4. Circuit statistics and comparison CSV")
print("\nNote: Circuit design only; model training and accuracy")
print("are not calculated by this code.")

PERSON 2 - QUANTUM CIRCUIT DESIGN

DATA VALIDATION
Training shape: (80, 2)
Testing shape : (20, 2)
Training labels: (80,)
Testing labels : (20,)
Angle range: 0.0 to 3.141592653589793

2-QUBIT NON-ENTANGLED CIRCUIT
     ┌──────────┐┌──────────────┐
q_0: ┤ Ry(x[0]) ├┤ Ry(theta[0]) ├
     ├──────────┤├──────────────┤
q_1: ┤ Ry(x[1]) ├┤ Ry(theta[1]) ├
     └──────────┘└──────────────┘

2-QUBIT ENTANGLED CIRCUIT
     ┌──────────┐┌──────────────┐     
q_0: ┤ Ry(x[0]) ├┤ Ry(theta[0]) ├──■──
     ├──────────┤├──────────────┤┌─┴─┐
q_1: ┤ Ry(x[1]) ├┤ Ry(theta[1]) ├┤ X ├
     └──────────┘└──────────────┘└───┘

2-qubit circuit images saved.

4-QUBIT NON-ENTANGLED CIRCUIT
     ┌───────────┐┌───────────────┐
q_0: ┤ Ry(x4[0]) ├┤ Ry(theta4[0]) ├
     ├───────────┤├───────────────┤
q_1: ┤ Ry(x4[1]) ├┤ Ry(theta4[1]) ├
     ├───────────┤├───────────────┤
q_2: ┤ Ry(x4[2]) ├┤ Ry(theta4[2]) ├
     ├───────────┤├───────────────┤
q_3: ┤ Ry(x4[3]) ├┤ Ry(theta4[3]) ├
     └───────────┘└───────────────┘

4-QUBIT

In [ ]:
%pip install -q qiskit qiskit-aer pylatexenc numpy pandas matplotlib scikit-learn

In [ ]:

# ================================================================
# PERSON 2 - REMAINING QUANTUM CIRCUIT DESIGN WORK
# Depth comparison, parameter table, measurement, Pauli gates
# Circuit diagrams, gate statistics, and CSV reports
# ================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector

print("=" * 65)
print("PERSON 2 - REMAINING WORK")
print("=" * 65)


# ================================================================
# PART 1: CREATE CIRCUITS WITH DIFFERENT DEPTHS
# ================================================================

def create_circuit(n_qubits=2, depth=1, entangled=False):
    """
    Creates a parameterized quantum circuit.

    n_qubits: number of qubits
    depth: number of repeated trainable layers
    entangled: whether to add CNOT gates
    """

    if n_qubits < 1 or depth < 1:
        raise ValueError("Number of qubits and depth must be positive.")

    qc = QuantumCircuit(n_qubits)

    # Data encoding parameters
    x = ParameterVector("x", n_qubits)

    # Trainable parameters: one parameter per qubit per layer
    theta = ParameterVector(
        "theta",
        n_qubits * depth
    )

    # Feature map: encode input angles
    for q in range(n_qubits):
        qc.ry(x[q], q)

    # Ansatz: trainable layers
    for layer in range(depth):

        for q in range(n_qubits):
            qc.ry(theta[layer * n_qubits + q], q)

        # Entanglement pattern
        if entangled:
            for q in range(n_qubits - 1):
                qc.cx(q, q + 1)

    return qc


# Build 2-qubit circuits with different depths
circuits = {}

for depth in [1, 2, 3]:

    circuits[f"Non-entangled depth {depth}"] = create_circuit(
        n_qubits=2,
        depth=depth,
        entangled=False
    )

    circuits[f"Entangled depth {depth}"] = create_circuit(
        n_qubits=2,
        depth=depth,
        entangled=True
    )


print("\nPART 1: CIRCUITS WITH DIFFERENT DEPTHS")

for name, qc in circuits.items():
    print(f"\n{name}")
    print(qc.draw(output="text"))


# ================================================================
# PART 2: CIRCUIT STATISTICS
# ================================================================

def get_stats(qc):
    counts = qc.count_ops()

    one_qubit = sum(
        instruction.operation.num_qubits == 1
        for instruction in qc.data
    )

    two_qubit = sum(
        instruction.operation.num_qubits == 2
        for instruction in qc.data
    )

    return {
        "Qubits": qc.num_qubits,
        "Circuit Depth": qc.depth(),
        "Total Gates": sum(counts.values()),
        "Single-Qubit Gates": one_qubit,
        "Two-Qubit Gates": two_qubit,
        "CNOT Gates": counts.get("cx", 0),
        "RY Gates": counts.get("ry", 0)
    }


statistics_rows = []

for name, qc in circuits.items():
    row = {"Circuit": name}
    row.update(get_stats(qc))
    statistics_rows.append(row)

statistics_df = pd.DataFrame(statistics_rows)

print("\nPART 2: CIRCUIT STATISTICS")
print(statistics_df.to_string(index=False))

statistics_df.to_csv(
    "person2_depth_comparison.csv",
    index=False
)


# ================================================================
# PART 3: PARAMETER TABLE
# ================================================================

parameter_rows = [
    {
        "Parameter": "x[i]",
        "Type": "Input parameter",
        "Purpose": "Encodes a classical feature as an RY rotation angle",
        "Example": "x[0] encodes feature 1"
    },
    {
        "Parameter": "theta[i]",
        "Type": "Trainable parameter",
        "Purpose": "Adjustable RY rotation used by the ansatz",
        "Example": "theta[0] controls the first trainable rotation"
    },
    {
        "Parameter": "depth",
        "Type": "Architecture setting",
        "Purpose": "Number of repeated trainable layers",
        "Example": "Depth 2 has two trainable layers"
    },
    {
        "Parameter": "CNOT",
        "Type": "Quantum gate",
        "Purpose": "Connects qubits and can create entanglement",
        "Example": "CX(0, 1)"
    },
    {
        "Parameter": "Measurement",
        "Type": "Readout operation",
        "Purpose": "Obtains classical bit outcomes from qubits",
        "Example": "Measure qubit 0 into classical bit 0"
    }
]

parameter_df = pd.DataFrame(parameter_rows)

print("\nPART 3: PARAMETER TABLE")
print(parameter_df.to_string(index=False))

parameter_df.to_csv(
    "person2_parameter_table.csv",
    index=False
)


# ================================================================
# PART 4: MEASUREMENT DEMONSTRATION
# ================================================================

print("\nPART 4: MEASUREMENT CIRCUITS")

# Measurement circuit based on a non-entangled circuit
measurement_non = circuits["Non-entangled depth 1"].copy()
measurement_non.measure_all()

# Measurement circuit based on an entangled circuit
measurement_ent = circuits["Entangled depth 1"].copy()
measurement_ent.measure_all()

print("\nNon-entangled circuit with measurement:")
print(measurement_non.draw(output="text"))

print("\nEntangled circuit with measurement:")
print(measurement_ent.draw(output="text"))


# ================================================================
# PART 5: SUPERPOSITION DEMONSTRATION
# ================================================================

print("\nPART 5: SUPERPOSITION DEMONSTRATION")

superposition_circuit = QuantumCircuit(1, 1)

# Hadamard gate creates an equal superposition from |0>
superposition_circuit.h(0)

# Measure the qubit
superposition_circuit.measure(0, 0)

print(superposition_circuit.draw(output="text"))

print(
    "The H gate prepares an equal superposition of |0> and |1> "
    "from the initial state |0>."
)


# ================================================================
# PART 6: PAULI GATE DEMONSTRATIONS
# ================================================================

print("\nPART 6: PAULI GATES")

pauli_circuits = {}

# Separate examples so each gate is easy to identify
for gate_name in ["X", "Y", "Z"]:

    qc = QuantumCircuit(1, 1)

    if gate_name == "X":
        qc.x(0)
    elif gate_name == "Y":
        qc.y(0)
    elif gate_name == "Z":
        qc.z(0)

    qc.measure(0, 0)

    pauli_circuits[gate_name] = qc

    print(f"\nPauli-{gate_name} circuit:")
    print(qc.draw(output="text"))

print("\nNote: Pauli gates are demonstrated separately.")
print("They are not automatically added to the main classifier circuits.")


# ================================================================
# PART 7: 4-QUBIT CIRCUITS
# ================================================================

print("\nPART 7: FOUR-QUBIT DEMONSTRATION")

qc_4_non = create_circuit(
    n_qubits=4,
    depth=1,
    entangled=False
)

qc_4_ent = create_circuit(
    n_qubits=4,
    depth=1,
    entangled=True
)

print("\n4-qubit non-entangled circuit:")
print(qc_4_non.draw(output="text"))

print("\n4-qubit entangled circuit:")
print(qc_4_ent.draw(output="text"))

four_qubit_df = pd.DataFrame([
    {"Circuit": "4-qubit non-entangled", **get_stats(qc_4_non)},
    {"Circuit": "4-qubit entangled", **get_stats(qc_4_ent)}
])

print("\nFour-qubit statistics:")
print(four_qubit_df.to_string(index=False))

four_qubit_df.to_csv(
    "person2_four_qubit_comparison.csv",
    index=False
)


# ================================================================
# PART 8: SAVE CIRCUIT DIAGRAM IMAGES
# ================================================================

print("\nPART 8: SAVING CIRCUIT IMAGES")

try:
    for name, qc in circuits.items():

        safe_name = (
            name.lower()
            .replace(" ", "_")
            .replace("-", "")
        )

        fig = qc.draw(output="mpl")

        fig.savefig(
            f"{safe_name}.png",
            dpi=200,
            bbox_inches="tight"
        )

        plt.close(fig)

    for name, qc in [
        ("measurement_non_entangled", measurement_non),
        ("measurement_entangled", measurement_ent),
        ("superposition", superposition_circuit),
        ("four_qubit_non_entangled", qc_4_non),
        ("four_qubit_entangled", qc_4_ent)
    ]:

        fig = qc.draw(output="mpl")

        fig.savefig(
            f"{name}.png",
            dpi=200,
            bbox_inches="tight"
        )

        plt.close(fig)

    print("Circuit images saved successfully.")

except Exception as error:
    print("Image drawing was unavailable:", error)
    print("Text diagrams were printed above.")
    print("CSV tables will still be saved.")


# ================================================================
# PART 9: FINAL SUMMARY
# ================================================================

print("\n" + "=" * 65)
print("PERSON 2 REMAINING WORK COMPLETED")
print("=" * 65)

print("""
Generated CSV files:
1. person2_depth_comparison.csv
2. person2_parameter_table.csv
3. person2_four_qubit_comparison.csv

Completed:
- Different circuit depths (1, 2, 3)
- Entangled and non-entangled circuit comparisons
- Gate counts and circuit depth
- Two-qubit gate and CNOT counts
- Parameter table
- Measurement demonstrations
- Superposition demonstration
- Pauli X, Y and Z demonstrations
- Four-qubit circuit demonstrations
- Circuit diagram image generation

Important:
These circuits are design demonstrations. The trainable parameters
have not been optimized, and no classification accuracy is calculated.
The four-qubit circuits are not trained on four-feature Iris data.
""")

PERSON 2 - REMAINING WORK

PART 1: CIRCUITS WITH DIFFERENT DEPTHS

Non-entangled depth 1
     ┌──────────┐┌──────────────┐
q_0: ┤ Ry(x[0]) ├┤ Ry(theta[0]) ├
     ├──────────┤├──────────────┤
q_1: ┤ Ry(x[1]) ├┤ Ry(theta[1]) ├
     └──────────┘└──────────────┘

Entangled depth 1
     ┌──────────┐┌──────────────┐     
q_0: ┤ Ry(x[0]) ├┤ Ry(theta[0]) ├──■──
     ├──────────┤├──────────────┤┌─┴─┐
q_1: ┤ Ry(x[1]) ├┤ Ry(theta[1]) ├┤ X ├
     └──────────┘└──────────────┘└───┘

Non-entangled depth 2
     ┌──────────┐┌──────────────┐┌──────────────┐
q_0: ┤ Ry(x[0]) ├┤ Ry(theta[0]) ├┤ Ry(theta[2]) ├
     ├──────────┤├──────────────┤├──────────────┤
q_1: ┤ Ry(x[1]) ├┤ Ry(theta[1]) ├┤ Ry(theta[3]) ├
     └──────────┘└──────────────┘└──────────────┘

Entangled depth 2
     ┌──────────┐┌──────────────┐     ┌──────────────┐     
q_0: ┤ Ry(x[0]) ├┤ Ry(theta[0]) ├──■──┤ Ry(theta[2]) ├──■──
     ├──────────┤├──────────────┤┌─┴─┐├──────────────┤┌─┴─┐
q_1: ┤ Ry(x[1]) ├┤ Ry(theta[1]) ├┤ X ├┤ Ry(theta[3]) ├┤